In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q17/annotated/checkpoints/pre_cell_2.pickle

trying: ['part']
me:  2
trying: ['lineitem']


me:  1
trying: ['pd']
me:  0
trying: ['load_part']
me:  2
trying: ['STORAGE_OPTS']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['load_lineitem']
me:  1
trying: ['tpch_parent']
me:  0


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable part
Declaring variable load_part


In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Cell 2 optimized

# 1. Filter PART to get the relevant keys on GPU
filtered_keys = part.query(
    "P_BRAND == 'Brand#23' & P_CONTAINER == 'MED BOX'"
).P_PARTKEY

# 2. Compute the 20%‐scaled average quantity per partkey on GPU
avg_map = lineitem.groupby("L_PARTKEY")["L_QUANTITY"].mean() * 0.2

# 3. Filter LINEITEM by those keys and map in the precomputed avg
li = lineitem[["L_PARTKEY", "L_QUANTITY", "L_EXTENDEDPRICE"]]
li = (
    li[li.L_PARTKEY.isin(filtered_keys)]
      .assign(avg=lambda df: df.L_PARTKEY.map(avg_map))
)

# 4. Apply the quantity filter and compute the final metric on GPU
filtered = li[li.L_QUANTITY < li.avg]
total = pd.DataFrame({
    "avg_yearly": [filtered.L_EXTENDEDPRICE.sum() / 7.0]
})

CPU times: user 16.6 s, sys: 2.01 s, total: 18.6 s
Wall time: 18.6 s


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q17/rewritten/o4_mini_high/checkpoints/post_cell_2_try_1.pickle

migration speed (bps): 807804673.0012894
---------------------------
variables to migrate:
total 48
lineitem 48
load_lineitem 144
STORAGE_OPTS 64
filtered_keys 48
load_part 144
pd 72
DATA_ROOT 80
li 48
tpch_parent 54
part 48
filtered 48
avg_map 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q17/rewritten/o4_mini_high/checkpoints/post_cell_2_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['filtered', 'total', 'part', 'avg_map', 'lineitem', 'li', 'filtered_keys']
Future variables []
Modified dataframes
  lineitem
    Input columns: set()
    Changed columns: {'L_QUANTITY', 'L_LINESTATUS', 'L_COMMENT', 'L_SUPPKEY', 'L_RECEIPTDATE', 'L_PARTKEY', 'L_DUMMY', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG', 'L_SHIPDATE', 'L_ORDERKEY', 'L_COMMITDATE', 'L_SHIPMODE', 'L_EXTENDEDPRICE', 'L_SHIPINSTRUCT', 'L_LINENUMBER'}
    Created columns: set()
    Deleted columns: set()
  part
    Input columns: set()
    Changed columns: {'P_CONTAINER', 'P_RETAILPRICE', 'P_COMMENT', 'P_SIZE', 'P_BRAND', 'P_PARTKEY', 'P_MFGR', 'P_TYPE', 'P_NA

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q17/opt_cell_exec_info_2_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q17/annotated/checkpoints/post_cell_2.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['tpch_parent']
me:  0
trying: ['pd']
me:  0
trying: ['total']
me:  3
trying: ['DATA_ROOT']
me:  0
trying: ['right']
me:  3
trying: ['lineitem']


me:  1
trying: ['STORAGE_OPTS']
me:  0
trying: ['load_lineitem']
me:  1
trying: ['left']
me:  3
trying: ['part']
me:  2
trying: ['line_part_merge']
me:  3
trying: ['load_part']
me:  2
trying: ['lineitem_filtered']


me:  3
trying: ['lineitem_avg']
me:  3
trying: ['orig_output']
me:  4


Declaring variable tpch_parent
Declaring variable pd
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable part
Declaring variable load_part
Declaring variable total
Declaring variable right
Declaring variable left
Declaring variable line_part_merge
Declaring variable lineitem_filtered
Declaring variable lineitem_avg
Declaring variable orig_output
